In [1]:
import os
if 'experiments' in os.getcwd ():
    os.chdir (os.getcwd () + "/..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
####### DATA PATHS #######
TRAIN_PATH = "./data/in/cattle_data_train.csv"
TEST_PATH = "./data/in/cattle_data_test.csv"
OUT_PATH = "./data/out/nnet.csv"

# Preprocessing

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option ("display.max_columns", None)
TARGET_FEATURE = "Milk_Yield_L"

DROP_FEATURES = [
    "Cattle_ID",
    "Farm_ID",
    "Feed_Quantity_lb",
    "Breed",
    "Climate_Zone",
    "Management_System",
    "Feed_Type",
    "Feeding_Frequency",
    "Walking_Distance_km",
    "Grazing_Duration_hrs",
    "Rumination_Time_hrs",
    "Resting_Hours",
    "Body_Condition_Score",
    "Humidity_percent",
    "BVD_Vaccine",
    "FMD_Vaccine",
    "Brucellosis_Vaccine",
    "HS_Vaccine",
    "BQ_Vaccine",
    "Housing_Score",
]

CATEGORICAL_FEATURES = [
    "Lactation_Stage",
    "Date",
    "Milking_Interval_hrs",
]

STANDARD_SCALED_FEATURES = [
    "Age_Months",
    "Weight_kg",
    "Parity",
    "Days_in_Milk",
    "Feed_Quantity_kg",
    "Water_Intake_L",
    "Ambient_Temperature_C",
    "Previous_Week_Avg_Yield",
]

def preprocess (dtrain, dtest):
    # Convert month to season
    def month_to_season (m):
        if m in [12, 1, 2]:
            return "Winter"
        elif m in [3, 4, 5]:
            return "Spring"
        elif m in [6, 7, 8]:
            return "Summer"
        else:
            return "Fall"

    months = pd.to_datetime (dtest['Date']).dt.month
    dtest = dtest.drop (columns = ['Date'])
    dtest['Date'] = months.apply (month_to_season)

    months = pd.to_datetime (dtrain['Date']).dt.month
    dtrain = dtrain.drop (columns = ['Date'])
    dtrain['Date'] = months.apply (month_to_season)

    # Imputation
    median_val = dtrain["Feed_Quantity_kg"].median ()

    dtrain.loc[dtrain["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val
    dtest.loc[dtest["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val

    # Square feed
    dtrain["Feed_Quantity_kg"] **= 2
    dtest["Feed_Quantity_kg"] **= 2

    # Drop features deemed unnecessary
    dtrain = dtrain.drop (DROP_FEATURES, axis = 1)
    dtest = dtest.drop (DROP_FEATURES, axis = 1)

    # One-hot encode
    dtrain = pd.get_dummies (dtrain, columns = CATEGORICAL_FEATURES, 
                             drop_first = True)
    dtest = pd.get_dummies (dtest, columns = CATEGORICAL_FEATURES, 
                            drop_first = True)
    dtrain, dtest = dtrain.align (dtest, join = 'left', axis = 1, fill_value = 0)

    # TODO: Do something about unreasonable data records
    # dtrain = dtrain.clip (lower = 0)
    # dtest = dtest.clip (lower = 0)

    # Standardize data
    scaler = StandardScaler ()
    # TODO  For now, standardize everything. In future, see if min/max scaling is better for non-gaussian data
    dtrain[STANDARD_SCALED_FEATURES] = scaler.fit_transform (
                                                dtrain[STANDARD_SCALED_FEATURES])
    dtest[STANDARD_SCALED_FEATURES] = scaler.transform (
                                                dtest[STANDARD_SCALED_FEATURES])

    return dtrain, dtest, scaler

train_data = pd.read_csv (TRAIN_PATH)
X_train, X_test, y_train, y_test = train_test_split (
    train_data.drop (TARGET_FEATURE, axis = 1),
    train_data[TARGET_FEATURE], test_size = 0.2, random_state = 0)

# TODO Should we scale the predicted milk yield too? and then apply the reverse of the scaler (inverse_transform)?

X_train, X_test, scaler = preprocess (X_train, X_test)

nan_cols = X_train.columns[X_train.isna ().any ()]

# Training Set Eval

In [4]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

# RMSE at each checkpoint
train_rmse_list = []
test_rmse_list = []

total_iterations = 200
iterations_per_step = 10

model = MLPRegressor (
    hidden_layer_sizes = (128, 96, 64),
    activation = "tanh",
    learning_rate_init = 0.00003,
    learning_rate = "adaptive",
    # alpha = 0.001,
    early_stopping = False,
    # validation_fraction = 0.15,
    n_iter_no_change = 20,
    verbose = False,
    warm_start = True,
    max_iter = iterations_per_step,
    random_state = 0)

for i in range (iterations_per_step, total_iterations+1, iterations_per_step):
    model.fit (X_train, y_train)

    # Compute RMSE
    y_train_pred = model.predict (X_train)
    y_test_pred = model.predict (X_test)
    train_rmse = np.sqrt (mean_squared_error (y_train, y_train_pred))
    test_rmse = np.sqrt (mean_squared_error (y_test, y_test_pred))

    train_rmse_list.append (train_rmse)
    test_rmse_list.append (test_rmse)

    print (f"Iteration {i}: Train RMSE={train_rmse:.4f}, Test RMSE={test_rmse:.4f}")

/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 10: Train RMSE=4.2646, Test RMSE=4.2711


/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 20: Train RMSE=4.1811, Test RMSE=4.1815


/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


...

Iteration 180: Train RMSE=4.1008, Test RMSE=4.1105


/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 190: Train RMSE=4.0998, Test RMSE=4.1104


/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 200: Train RMSE=4.0989, Test RMSE=4.1103


In [5]:
# Predict on all data for feedback
raw_data = pd.read_csv (TRAIN_PATH)

y_pred = model.predict (X_train)
rmse = np.sqrt (mean_squared_error (y_train, y_pred))
print ("Train RMSE:", rmse)

y_pred = model.predict (X_test)
rmse = np.sqrt (mean_squared_error (y_test, y_pred))
print ("Test RMSE:", rmse)

# See what was good and bad
errors = np.sqrt ((y_test - y_pred) ** 2)
df_results = X_test.copy ()
scaled_part = df_results[STANDARD_SCALED_FEATURES]
scaled_inverse = pd.DataFrame (scaler.inverse_transform (scaled_part),
                               columns = STANDARD_SCALED_FEATURES,
                               index = df_results.index,)

# Replace only those columns
df_results[STANDARD_SCALED_FEATURES] = scaled_inverse
df_results["y_true"] = y_test
df_results["y_pred"] = y_pred
df_results["rmse"] = errors

df_results = df_results.merge(
    raw_data,     # <-- this contains ALL original columns
    left_index=True,
    right_index=True,
    how="left"
)

print ("\nTop 5 BEST predictions:")
print(df_results.nsmallest (5, "rmse"))

print ("\nTop 5 WORST predictions:")
print (df_results.nlargest (5, "rmse"))

Train RMSE: 4.0988704662942075
Test RMSE: 4.110341266907565

Top 5 BEST predictions:
        Age_Months_x  Weight_kg_x  Parity_x  Days_in_Milk_x  \
112861         103.0        573.1       4.0           340.0   
94283          135.0        588.5       5.0            30.0   
20435           52.0        641.3       4.0           352.0   
86471           68.0        312.0       1.0            79.0   
142199          97.0        638.6       6.0           203.0   

        Feed_Quantity_kg_x  Water_Intake_L_x  Ambient_Temperature_C_x  \
112861          172.931249         63.863680                22.730842   
94283           103.739983         82.741448                27.114249   
20435            29.027692        118.732334                10.519967   
86471            94.662642         65.397639                25.985856   
142199          143.253242         81.441065                16.715875   

        Anthrax_Vaccine_x  IBR_Vaccine_x  Rabies_Vaccine_x  \
112861                  1          

# Final Model

In [6]:
# Build final model
train_data = pd.read_csv (TRAIN_PATH)
test_data = pd.read_csv (TEST_PATH)

X_train = train_data.drop (TARGET_FEATURE, axis = 1)
y_train = train_data[TARGET_FEATURE]
X_test = test_data

X_train, X_test, scaler = preprocess (X_train, X_test)

# TODO MAKE SURE THAT THESE ARE THE SAME AS THE TRAINED ONE
TARGET_ITERATIONS = 200
model = MLPRegressor (
    hidden_layer_sizes = (128, 96, 64),
    activation = "tanh",
    learning_rate_init = 0.00003,
    learning_rate = "adaptive",
    # alpha = 0.001,
    early_stopping = False,
    # validation_fraction = 0.15,
    n_iter_no_change = 20,
    verbose = False,
    warm_start = True,
    max_iter = TARGET_ITERATIONS,
    random_state = 0)

model.fit (X_train, y_train)

/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,loss,'squared_error'
,hidden_layer_sizes,"(128, ...)"
,activation,'tanh'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'adaptive'
,learning_rate_init,3e-05
,power_t,0.5
,max_iter,200
,shuffle,True


In [7]:
# Final Predictions
y_pred = model.predict (X_test)

print (y_pred.mean ())
out_data = pd.DataFrame ({'Cattle_ID': np.arange (1, len (y_pred) + 1),
                          'Milk_Yield_L': y_pred})
out_data.to_csv (OUT_PATH, index = False)

15.561907402240404
